## Week 1 · Data augmentation  🎨

Deep-learning models generalise better when they see **a lot of varied data**, but recording and labelling frames is slow. **Data augmentation** allows you tou increase the number of available samples by creating modified copies of them: flipping, cropping, blurring, changing brightness, shifting colours...

Each copy looks different to the model yet shows the same scene, so the model learns to be **robust** to changes in viewpoint, lighting and image quality instead of memorising the training frames. This usually improves accuracy and reduces overfitting at no extra labelling cost.

Two things to keep in mind:
- **The labels move with the image.** When a frame is flipped or cropped its bounding boxes must be transformed too. The [augmentation library](https://albumentations.ai/explore/) does this for us.
- **Only augment real data.** Augmenting already-augmented frames piles distortion on distortion without adding information, so we always start from the original sequences. The notebook does this for us.

Augmented frames are written to a new `<sequence>-aug/` folder next to each original sequence:

```
dataset/
├── Video_1/       # original sequence
│   ├── images/
│   └── labels/
└── Video_1-aug/   # augmented copies
    ├── images/
    └── labels/
```

### Imports

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET

from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import albumentations as alb

### Settings

- `PATH_DATA`: folder that holds your sequences.
- `INPUT_SIZE`: `(width, height)` every augmented frame is resized to.

In [ ]:
PATH_DATA = '../dataset'      # Folder with the labelled sequences
INPUT_SIZE = (1280, 720)      # (width, height) of the augmented frames

### Load the dataset

Read every labelled frame into a list of samples `{'frame_path', 'bboxes', 'labels'}`. Already-augmented sequences are skipped so they are never used as a source.

In [ ]:
def list_dir(path):
    """List a folder, skipping caches like __pycache__ and .ipynb_checkpoints."""
    return sorted(e for e in os.listdir(path) if not e.startswith('.') and e != '__pycache__')


def load_labels(xml_path):
    """Read a Pascal VOC .xml file, returning (bboxes, labels)."""
    root = ET.parse(xml_path).getroot()
    bboxes, labels = [], []
    for obj in root.iter('object'):
        box = obj.find('bndbox')
        bboxes.append([int(box.find(c).text) for c in ('xmin', 'ymin', 'xmax', 'ymax')])
        labels.append(obj.find('name').text)
    return bboxes, labels


def load_data(path):
    """Load every labelled frame as {'frame_path', 'bboxes', 'labels'} (augmented sequences excluded)."""
    samples = []
    for seq in list_dir(path):
        seq_dir = os.path.join(path, seq)
        if not os.path.isdir(seq_dir) or seq.endswith('-aug'):
            continue
        labels_dir = os.path.join(seq_dir, 'labels')
        for label_file in list_dir(labels_dir):
            name = os.path.splitext(label_file)[0]
            bboxes, labels = load_labels(os.path.join(labels_dir, label_file))
            samples.append({
                'frame_path': os.path.join(seq_dir, 'images', f'{name}.jpg'),
                'bboxes': bboxes,
                'labels': labels,
            })
    return samples

In [ ]:
samples = load_data(PATH_DATA)
print(f'Loaded {len(samples)} labelled frames')

Loaded 605 labelled frames


### Preview frames and boxes

Draw a few frames with their bounding boxes to check the labels look right before augmenting.

In [ ]:
def draw_frame(ax, image, bboxes, color='tab:green'):
    """Show an image on `ax` with its bounding boxes as coloured outlines."""
    ax.imshow(image)
    ax.axis('off')
    for x_min, y_min, x_max, y_max in bboxes:
        ax.add_patch(Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                               linewidth=2, edgecolor=color, facecolor='none'))


def show_batch(samples, n=4):
    """Display the first n samples with their bounding boxes."""
    samples = samples[:n]
    fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4), squeeze=False)
    for ax, sample in zip(axes.ravel(), samples):
        draw_frame(ax, plt.imread(sample['frame_path']), sample['bboxes'])
    plt.tight_layout()
    plt.show()

In [ ]:
show_batch(samples)

<Figure size 1600x400 with 4 Axes>

### Create the augmentor

The augmentor is an [albumentations](https://albumentations.ai/explore/) pipeline: a list of transforms, each applied with some probability `p`. **This is the cell to play with** — comment transforms out, change their parameters or add new ones to see how the augmentations change.

`bbox_params` tells albumentations to transform the bounding boxes together with the image and to drop boxes that become too small or barely visible after the transform.

In [ ]:
def create_augmentor():
    """Build the albumentations augmentation pipeline (edit the transforms here)."""
    width, height = INPUT_SIZE
    return alb.Compose(
        [
            alb.HorizontalFlip(p=0.5),
            alb.Blur(blur_limit=7, p=0.5),
            alb.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.2),
            alb.RandomGamma(gamma_limit=(80, 120), p=0.2),
            alb.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.2),
            alb.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=10, p=0.5),
            alb.RandomCrop(height=480, width=640, p=0.5),
            alb.Resize(height=height, width=width, p=1.0),
        ],
        bbox_params=alb.BboxParams(format='pascal_voc', min_area=100, min_visibility=0.3, label_fields=['labels']),
    )

In [ ]:
augmentor = create_augmentor()

### Preview an augmentation

Apply the augmentor a few times to the same frame. The scene stays recognisable while colours, framing and sharpness change, and the boxes follow the image.

In [ ]:
def show_augmentation(augmentor, sample, n=3):
    """Show an original frame next to n augmented versions of it."""
    image = plt.imread(sample['frame_path'])
    fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4), squeeze=False)
    draw_frame(axes[0, 0], image, sample['bboxes'], color='tab:blue')
    axes[0, 0].set_title('Original')
    for ax in axes[0, 1:]:
        aug = augmentor(image=image, bboxes=sample['bboxes'], labels=sample['labels'])
        draw_frame(ax, aug['image'], aug['bboxes'])
        ax.set_title('Augmented')
    plt.tight_layout()
    plt.show()

In [ ]:
show_augmentation(augmentor, samples[0])

<Figure size 1600x400 with 4 Axes>

### Run data augmentation

For every original sequence, create a sibling `<sequence>-aug/` folder and write `num_augments` augmented copies of each frame (with their transformed labels).

To avoid piling up synthetic data, `-aug` folders are never used as a source, and if a sequence has already been augmented it is **skipped** — set `replace=True` to regenerate it from scratch instead.

In [ ]:
def save_labels(xml_path, bboxes, labels):
    """Write bounding boxes and labels to a Pascal VOC .xml file."""
    width, height = INPUT_SIZE
    filename = f'{os.path.splitext(os.path.basename(xml_path))[0]}.jpg'
    objects = ''.join(
        f"""
    <object>
        <name>{label}</name>
        <bndbox>
            <xmin>{int(x_min)}</xmin>
            <ymin>{int(y_min)}</ymin>
            <xmax>{int(x_max)}</xmax>
            <ymax>{int(y_max)}</ymax>
        </bndbox>
    </object>"""
        for (x_min, y_min, x_max, y_max), label in zip(bboxes, labels)
    )
    with open(xml_path, 'w') as f:
        f.write(f"""<annotation>
    <folder>images</folder>
    <filename>{filename}</filename>
    <size>
        <width>{width}</width>
        <height>{height}</height>
        <depth>3</depth>
    </size>{objects}
</annotation>
""")


def augment_sequence(augmentor, src_dir, dst_dir, num_augments):
    """Augment every labelled frame in src_dir num_augments times into dst_dir."""
    os.makedirs(os.path.join(dst_dir, 'images'))
    os.makedirs(os.path.join(dst_dir, 'labels'))
    for label_file in tqdm(list_dir(os.path.join(src_dir, 'labels')), desc=os.path.basename(dst_dir)):
        name = os.path.splitext(label_file)[0]
        image = plt.imread(os.path.join(src_dir, 'images', f'{name}.jpg'))
        bboxes, labels = load_labels(os.path.join(src_dir, 'labels', label_file))
        for i in range(num_augments):
            aug = augmentor(image=image, bboxes=bboxes, labels=labels)
            stem = f'{name}_aug_{i:02d}'
            plt.imsave(os.path.join(dst_dir, 'images', f'{stem}.jpg'), aug['image'])
            save_labels(os.path.join(dst_dir, 'labels', f'{stem}.xml'), aug['bboxes'], aug['labels'])


def run_data_augmentation(augmentor, num_augments=5, replace=False):
    """Augment every original sequence into a sibling '<sequence>-aug' folder.

    Args:
        augmentor: pipeline from create_augmentor().
        num_augments: number of augmented copies per frame.
        replace: regenerate a sequence that was already augmented (True) or skip it (False).
    """
    for seq in list_dir(PATH_DATA):
        src_dir = os.path.join(PATH_DATA, seq)
        # Only augment real sequences: skip files and already-augmented folders
        if not os.path.isdir(src_dir) or seq.endswith('-aug'):
            continue

        dst_dir = os.path.join(PATH_DATA, f'{seq}-aug')
        if os.path.exists(dst_dir):
            if not replace:
                print(f'Skipping {seq}: {seq}-aug already exists (use replace=True to regenerate)')
                continue
            print(f'Warning: replacing {seq}-aug')
            shutil.rmtree(dst_dir)

        augment_sequence(augmentor, src_dir, dst_dir, num_augments)

In [15]:
run_data_augmentation(augmentor, num_augments=2, replace=True)

RoboMaster-Aisle-aug:   0%|          | 0/129 [00:00<?, ?it/s]

RoboMaster-Aisle-aug: 100%|██████████| 129/129 [00:02<00:00, 49.55it/s]


RoboMaster-GTI-aug: 100%|██████████| 122/122 [00:02<00:00, 47.66it/s]


RoboMaster-Parking-aug: 100%|██████████| 126/126 [00:02<00:00, 46.96it/s]


RoboMaster-SoccerCourt-aug: 100%|██████████| 120/120 [00:02<00:00, 48.19it/s]


RoboMaster-Stairs-aug: 100%|██████████| 108/108 [00:02<00:00, 38.95it/s]
